# Upper-celerity cap investigation — 20 October 2026

**Separate diagnostics only. All pre-existing results and active calibration files remain unchanged.**

We audit the 5 m/s upper cap across all saved states, then reroute the parent and segmented collapse for each of the 15 examples. Three scenarios use a common refined grid and timestep within each example: capped control; uncapped at current calibration; uncapped with conditional roughness recalibration. The lower celerity bound is retained.

The uncapped ceiling is above every raw celerity throughout the declared roughness range. Physical link lengths and geometry are preserved. Grid refinement is necessary where removing the cap violates the existing spatial bound; the matching capped control isolates this numerical change.

The conditional calibration adjusts **only Manning n**, preserving beta, the archived calibration forcing, targets and crest operator. It is not a full joint calibration or a validated replacement model. Example 54 retains its archived calibration operator; response peak timings still use global maxima.

All response comparisons use the existing shared 0.5% analysis-window boundaries. A single baseline is shared across diagnostic states and scenarios; any required baseline change is explicitly reported below. Example 54's finite-window distribution includes its transition to a higher ending equilibrium.

In [2]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'RAPID').exists())
out = root/'RAPID/outputs/cap_investigation_20261020'
display(json.loads((out/'summary.json').read_text()))
display(json.loads((out/'preservation_check.json').read_text()))

{'states_audited': 2217, 'states_with_active_upper_cap': 1154, 'networks_with_active_cap': ['13', '15', '21', '22', '46', '48', '54'], 'routed_scenarios': 90, 'all_routes_settled': True, 'minimum_native_flow_cms': 1.054301551762358, 'baseline_changed': False, 'preserved_files': 3599, 'scope': 'All saved states audited analytically and for grid preparation; actual rerouting limited to 15 parent/segmented pairs. Conditional recalibration varies n only, with beta fixed. No replacement production outputs.'}

{'files_checked': 3599, 'changed': [], 'passed': True}

## Where the cap is active

The audit's `change_in_collapse_shift_hours` is an analytic mean-route-time contrast, **not a simulated centroid shift**. It is useful for screening; actual hydrograph responses appear separately below.

In [4]:
audit = pd.read_csv(out/'network_cap_audit.csv', dtype={'example':str})
display(audit)
routes = pd.read_csv(out/'routing/metrics.csv', dtype={'example':str})
summary = pd.read_csv(out/'segmented_centroid_summary.csv', dtype={'example':str})
display(summary)
px.bar(summary, x='example', y='uncapping_effect_hours',
       labels={'uncapping_effect_hours':'Uncapped − capped centroid shift (hours)'},
       title='Effect of removing the cap at unchanged calibration').show()

,example,states,affected_states,max_raw_celerity,max_abs_shift_effect_hours
0,02,38,0,2.299221,0.000000
1,03,43,0,1.738468,0.000000
2,05,54,0,1.858043,0.000000
3,13,69,65,5.358423,0.001249
4,15,265,264,6.407998,0.370613
5,21,33,32,6.919716,0.279574
6,22,411,411,13.500030,5.084984
7,37,399,0,4.422397,0.000000
8,41,185,0,4.817718,0.000000
9,45,96,0,2.884271,0.000000


,example,capped_control,uncapped,uncapped_refit,uncapping_effect_hours
0,02,-0.965527,-0.965527,-0.965527,0.000000
1,03,-2.179386,-2.179386,-2.179386,0.000000
2,05,-6.534990,-6.534990,-6.534990,0.000000
3,13,-6.980202,-6.979301,-6.979301,0.000901
4,15,-4.459603,-4.768222,-4.768222,-0.308619
5,21,-1.727586,-1.745648,-1.772321,-0.018062
6,22,-5.263221,-8.243433,-8.930121,-2.980212
7,37,-7.764332,-7.764332,-7.764332,0.000000
8,41,-13.244369,-13.244369,-13.244369,0.000000
9,45,-8.753746,-8.753746,-8.753746,0.000000


## Conditional recalibration

The original timing target can be restored by changing roughness in the affected parents. This does not imply unchanged collapse responses: both parent and child receive the recalibrated roughness, and their geometry differs. Peak-height interval distances are reported because recovering timing alone may worsen magnitude agreement.

In [6]:
records = json.loads((out/'conditional_recalibration/results.json').read_text())
display(pd.DataFrame([{
    'example':r['example'], 'old_n':r['old_n'], 'new_n':r['new_n'],
    'uncapped_old_n_timing_error_minutes':r['uncapped_old_n']['timing_error_minutes'],
    'refitted_timing_error_minutes':r['selected']['timing_error_minutes'],
    'old_height_distance_cms':r['old_calibration_height_distance_cms'],
    'new_height_distance_cms':r['selected']['height_distance_cms']
} for r in records if r['refitted']]))
display(routes[['example','state','scenario','centroid_shift_hours','peak_shift_hours',
                'minimum_native_flow_cms','settled','max_absolute_difference_from_published_cms']])

,example,old_n,new_n,uncapped_old_n_timing_error_minutes,refitted_timing_error_minutes,old_height_distance_cms,new_height_distance_cms
0,13,0.019666,0.019666,0.000000,0.000000,0.000000,0.000000
1,15,0.014309,0.014309,0.366667,0.366667,0.000000,0.000000
2,21,0.023948,0.024340,-17.500000,0.500000,15.594887,22.695066
3,22,0.014721,0.016240,-124.500000,0.500000,144.130166,168.909423
4,46,0.017839,0.017896,-19.500000,0.500000,273.721142,275.306387
5,48,0.023573,0.023720,-2.500000,0.500000,0.000000,0.000000
6,54,0.030023,0.030023,0.000000,0.000000,1.946509,1.933825


,example,state,scenario,centroid_shift_hours,peak_shift_hours,minimum_native_flow_cms,settled,max_absolute_difference_from_published_cms
0,02,S000_base,capped_control,0.000000,0.000000,11.639888,True,0.000000
1,02,S000_base,uncapped,0.000000,0.000000,11.639888,True,0.000000
2,02,S000_base,uncapped_refit,0.000000,0.000000,11.639888,True,0.000000
3,02,all_closed_units_collapsed,capped_control,-0.965527,-1.683333,946.986290,True,0.000000
4,02,all_closed_units_collapsed,uncapped,-0.965527,-1.683333,946.986290,True,0.000000
...,...,...,...,...,...,...,...,...
85,56,S000_base,uncapped,0.000000,0.000000,33.890269,True,0.000000
86,56,S000_base,uncapped_refit,0.000000,0.000000,33.890269,True,0.000000
87,56,all_closed_units_collapsed,capped_control,-2.537610,-2.800000,661.738535,True,0.000784
88,56,all_closed_units_collapsed,uncapped,-2.537610,-2.800000,661.738535,True,0.000784


## Interpretation and limits

The upper cap is not negligible under the current calibration, especially for example 22. Stable, nonnegative uncapped routes on appropriately refined grids demonstrate numerical feasibility for the tested parent/segmented states; they do not establish physical credibility of raw celerities approaching 13 m/s, or guarantee every partial collapse will route acceptably.

Example 22's segmented centroid response changes from approximately −5.26 h to −8.24 h at fixed calibration and −8.93 h after conditional timing recalibration. Restoring parent timing does not restore the original collapse response. Other networks are substantially less sensitive.

Keep the frozen production results unchanged. Before choosing an uncapped replacement, independently assess the prescribed depth, slope and roughness assumptions, and perform a complete, consistent calibration and response regeneration. Do not remove the cap solely because it moves one case closer to a predictor's estimate. The remaining peak differences between matching-grid controls and published curves should also be distinguished from the much more stable centroid.

This investigation does not refit the forest, alter predictors, or assess its predictive scores on changed targets.